In [4]:
import pandas as pd     # Импорт библиотеки pandas для работы с табличными данными (DataFrame)
import numpy as np      # Импорт библиотеки numpy для работы с массивами и числовыми операциями
import seaborn as sns   # Импорт библиотеки seaborn для создания красивых и информативных визуализаций данных
from matplotlib import pyplot as plt              # Импорт функции pyplot из библиотеки matplotlib для создания графиков и визуализаций

from sklearn.preprocessing import OneHotEncoder, StandardScaler       # Импорт классов для кодирования категориальных признаков и стандартизации данных
from sklearn.model_selection import train_test_split   # Импорт функций для разбиения данных на тренировочную и тестовую выборки

from sklearn.linear_model import LogisticRegression   # Импорт модели логистической регрессии
from sklearn.neighbors import KNeighborsClassifier     # Импорт классификатора К-ближайших соседей
from sklearn.tree import DecisionTreeClassifier       # Импорт классификатора на основе дерева решений
from sklearn.svm import SVC                           # Импорт классификатора на основе метода опорных векторов (SVM)

from sklearn import metrics   # Импорт всего модуля metrics для работы с метриками оценки моделей
from sklearn.metrics import ( # Импорт конкретных функций оценки из подмодуля metrics
accuracy_score,            # Импорт функции для оценки точности (ассгасу) модели, т.е. доли правильно предсказанных примеров
balanced_accuracy_score,   # Импорт функции для оценки сбалансированной точности, учитывающей классовую неоднородность
precision_score,           # Импорт функции для вычисления точности (precision)
recall_score,              # Импорт функции для вычисления полноты (recall)
confusion_matrix,          # Импорт функции для создания матрицы ошибок (confusion matrix): TN, TP, FN, FP
multilabel_confusion_matrix, # Импорт функции для создания матрицы ошибок для случаев с многомерной классификацией
f1_score,                  # Импорт функции для вычисления F1-меры
roc_auc_score,             # Импорт функции для вычисления AUC-ROC (площадь под кривой приемлемости и ошибок)
roc_curve,                 # Импорт функции для вычисления координат для построения ROC-кривой
average_precision_score,   # Импорт функции для вычисления средней точности (average precision)
precision_recall_curve )   # Импорт функции для вычисления и визуализации кривой "точность-полнота" (precision-recall curve)


heart = pd.read_csv('/Users/konstantingeneralov/IDE/Data/heart_data.tsv', sep='\t')
heart_df = heart.copy()  # Создание копии DataFrame для дальнейшей работы
#дропнем столбец 'ID', так как он не несет полезной информации для анализа и моделирования
heart_df.drop(columns=['ID'], inplace=True)
display(heart_df.head(), heart_df.info(), heart_df.describe(include='all')) 


#Подсчитаем количество значений всех признаков датасета, включая пропущенные:


for col in heart_df.columns:
    print(f'Признак: {col}')
    print(heart_df[col].value_counts(dropna=False).to_string())

<class 'pandas.DataFrame'>
RangeIndex: 955 entries, 0 to 954
Data columns (total 37 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Пол                                     954 non-null    str    
 1   Семья                                   955 non-null    str    
 2   Этнос                                   955 non-null    str    
 3   Национальность                          955 non-null    str    
 4   Религия                                 955 non-null    str    
 5   Образование                             955 non-null    str    
 6   Профессия                               955 non-null    str    
 7   Работа                                  955 non-null    int64  
 8   Выход на пенсию                         955 non-null    int64  
 9   Прекращение работы по болезни           955 non-null    int64  
 10  Сахарный диабет                         955 non-null    int64  
 11  Гепа

,Пол,Семья,Этнос,Национальность,Религия,Образование,Профессия,Работа,Выход на пенсию,Прекращение работы по болезни,...,Время засыпания,Время пробуждения,Сон после обеда,"Спорт, клубы","Религия, клубы",Артериальная гипертензия,ОНМК,"Стенокардия, ИБС, инфаркт миокарда",Сердечная недостаточность,Прочие заболевания сердца
0,М,в браке в настоящее время,европейская,Русские,Христианство,3 - средняя школа / закон.среднее / выше среднего,низкоквалифицированные работники,1,0,0,...,22:00:00,06:00:00,0,0,0,0,0,0,0,0
1,Ж,в разводе,европейская,Русские,Христианство,5 - ВУЗ,дипломированные специалисты,0,0,0,...,00:00:00,04:00:00,1,0,0,1,1,0,0,0
2,Ж,в браке в настоящее время,европейская,Русские,Христианство,5 - ВУЗ,дипломированные специалисты,0,0,0,...,23:00:00,07:00:00,0,0,0,0,0,0,0,0
3,М,в браке в настоящее время,европейская,Русские,Атеист / агностик,3 - средняя школа / закон.среднее / выше среднего,низкоквалифицированные работники,1,0,0,...,23:00:00,07:00:00,0,0,0,1,0,0,0,0
4,Ж,в браке в настоящее время,европейская,Русские,Христианство,3 - средняя школа / закон.среднее / выше среднего,операторы и монтажники установок и машинного о...,0,0,1,...,23:00:00,06:00:00,0,0,0,1,0,1,1,0


None

,Пол,Семья,Этнос,Национальность,Религия,Образование,Профессия,Работа,Выход на пенсию,Прекращение работы по болезни,...,Время засыпания,Время пробуждения,Сон после обеда,"Спорт, клубы","Религия, клубы",Артериальная гипертензия,ОНМК,"Стенокардия, ИБС, инфаркт миокарда",Сердечная недостаточность,Прочие заболевания сердца
count,954,955,955,955,955,955,955,955.000000,955.000000,955.000000,...,955,955,955.000000,955.000000,955.000000,955.000000,955.000000,955.000000,955.000000,955.000000
unique,2,6,3,18,4,4,11,NaN,NaN,NaN,...,22,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Ж,в браке в настоящее время,европейская,Русские,Христианство,4 - профессиональное училище,дипломированные специалисты,NaN,NaN,NaN,...,23:00:00,06:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,667,563,931,899,804,454,225,NaN,NaN,NaN,...,311,233,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.523560,0.335079,0.048168,...,NaN,NaN,0.226178,0.068063,0.023037,0.467016,0.042932,0.122513,0.100524,0.090052
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.499706,0.472265,0.214232,...,NaN,NaN,0.418575,0.251986,0.150098,0.499172,0.202810,0.328049,0.300854,0.286407
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,0.000000,...,NaN,NaN,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000


Признак: Пол
Пол
Ж      667
М      287
NaN      1
Признак: Семья
Семья
в браке в настоящее время                          563
вдовец / вдова                                     143
в разводе                                          125
гражданский брак / проживание с партнером           79
никогда не был(а) в браке                           43
раздельное проживание (официально не разведены)      2
Признак: Этнос
Этнос
европейская                                                                                                      931
другая азиатская (Корея, Малайзия, Таиланд, Вьетнам, Казахстан, Киргизия, Туркмения, Узбекистан, Таджикистан)     17
прочее (любая иная этно-расовая группа, не представленная выше)                                                    7
Признак: Национальность
Национальность
Русские                  899
Татары                    18
Немцы                      6
Чуваши                     5
Азербайджанцы              4
Украинцы                   4
Другие национа

In [39]:
# Кодирование категориальных признаков One-Hot методом

heart_df_3 = pd.read_csv('/Users/konstantingeneralov/IDE/Data/heart_data_cleaned.csv', sep=',')
heart_4 = heart_df_3.copy()  # Создание копии DataFrame для дальнейшей работы

columns_to_change = ['Семья', 'Этнос', 'Национальность', 'Религия', 'Профессия', 'Статус Курения', 'Алкоголь']
for column in columns_to_change:
 print('Число уникальных значений признака {}: '.format(column), heart_4[column].nunique())

# инициализация кодировщика OneHotEncoder
one_hot_encoder = OneHotEncoder()

# Обучаем энкодер, применяем преобразование к выборке. Результат переводим в массив
encoded_features = one_hot_encoder.fit_transform(heart_4[columns_to_change]).toarray()

# Сохраним полученные названия новых колонок в переменную colums
colums = one_hot_encoder.get_feature_names_out(columns_to_change)

# Преобразуем массив в формат DataFrame
encoded_df = pd.DataFrame(encoded_features, columns=colums)

# Переустановим индексацию в таблицах
# reset_index() — изменяет индексы с рандомных на последовательные от 0 до n
# drop(['index'], axis = 1) — удаляет образовавшийся столбец 'index'
encoded_df.reset_index(drop=True, inplace=True)
heart_4.reset_index(drop=True, inplace=True)   

# Объединим таблицы
heart_4 = pd.concat([heart_4, encoded_df], axis=1)

# Удалим старые категориальные признаки
heart_4.drop(columns=columns_to_change, inplace=True)

# Сделаем все признаки целочисленными
heart_4 = heart_4.astype(int)

summary = pd.DataFrame({
    "missing": heart_4.isnull().sum(),
    "unique": heart_4.nunique(),
    "dtype": heart_4.dtypes
})
display(summary)

heart_5 = heart_4.copy()  # Создание копии DataFrame для дальнейшей работы

# Удалим одну из коррелирующих переменных в каждой паре:
# Статус Курения_Никогда не курил(а)
# Алкоголь_никогда не употреблял
# Этнос_другая азиатская (Корея, Малайзия, Таиланд, Вьетнам, Казахстан, Киргизия, Туркмения, Узбекистан, Таджикистан)
heart_5.drop(columns=['Статус Курения_Никогда не курил(а)', 'Алкоголь_никогда не употреблял', 
                      'Этнос_другая азиатская (Корея, Малайзия, Таиланд, Вьетнам, Казахстан, Киргизия, Туркмения, Узбекистан, Таджикистан)'], inplace=True)

# сохраним конечный результат в новый файл
heart_5.to_csv('/Users/konstantingeneralov/IDE/Data/heart_data_final.csv', index=False)

# display(heart_5.head(), heart_5.info(), heart_5.describe(include='all'))

Число уникальных значений признака Семья:  6
Число уникальных значений признака Этнос:  3
Число уникальных значений признака Национальность:  18
Число уникальных значений признака Религия:  4
Число уникальных значений признака Профессия:  11
Число уникальных значений признака Статус Курения:  3
Число уникальных значений признака Алкоголь:  3


,missing,unique,dtype
Пол,0,2,int64
Образование,0,4,int64
Работа,0,2,int64
Выход на пенсию,0,2,int64
Прекращение работы по болезни,0,2,int64
...,...,...,...
Статус Курения_Курит,0,2,int64
Статус Курения_Никогда не курил(а),0,2,int64
Алкоголь_никогда не употреблял,0,2,int64
Алкоголь_ранее употреблял,0,2,int64


In [43]:
#  Кодирование категориальных признаков по популярности

#display(heart_df_3.head(), heart_df_3.info(), heart_df_3.describe(include='all'))

heart_popularity = heart_df_3.copy()  # Создание копии DataFrame для дальнейшей работы

# Закодируем признаки по популярности с добавлением в закодированные значения небольшого уровня шума, чтобы различать категории с одинаковой популярностью

np.random.seed(42)
cats = ['Семья', 'Этнос', 'Национальность', 'Религия', 'Профессия', 'Статус Курения', 'Алкоголь']
for cat in cats:
    heart_popularity[cat] = heart_popularity[cat].map(heart_popularity[cat].value_counts()) + 0.1 * np.random.random(len(heart_popularity))

#Найдем сильно коррелирующие между собой признаки и удалим один из них в каждой паре, чтобы избежать мультиколлинеарности в модели
corr = heart_popularity.corr(numeric_only=True)
m = (corr.mask(np.eye(len(corr), dtype=bool)).abs() > 0.8).any()
raw = corr.loc[m, m]
raw

heart_popularity.drop(['Статус Курения'], axis=1, inplace=True)

# сохраним конечный результат в новый файл
heart_popularity.to_csv('/Users/konstantingeneralov/IDE/Data/heart_data_popularity.csv', index=False)

#display(heart_popularity.head(), heart_popularity.info(), heart_popularity.describe(include='all'))